# Sync Gold Tables to Lakebase

Creates synced tables to replicate gold layer data into Lakebase for low-latency app access.

**Prerequisites:**
- Gold tables exist with CDF enabled (pipeline must have run)
- Lakebase project `db-residential-copilot` is ONLINE (notebook 02)

**Synced tables created:**
- `dbx_res_gold.portfolio_metrics` ← `startups_catalog.dbx_res_gold.gold_portfolio_property_metrics`
- `dbx_res_gold.portfolio_time_series` ← `startups_catalog.dbx_res_gold.gold_portfolio_time_series`

In [ ]:
%pip install -U "databricks-sdk>=0.81.0"
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

PROJECT_ID = "db-residential-copilot"
CATALOG = "startups_catalog"
BRANCH = "production"
DATABASE = "databricks_postgres"
# UC catalog name for the synced tables (underscores, no hyphens)
LAKEBASE_CATALOG = "db_residential_copilot"

## Verify CDF is Enabled on Gold Tables

In [ ]:
gold_tables = [
    f"{CATALOG}.dbx_res_gold.gold_portfolio_property_metrics",
    f"{CATALOG}.dbx_res_gold.gold_portfolio_time_series",
]

for table in gold_tables:
    props = spark.sql(f"SHOW TBLPROPERTIES {table}").filter("key = 'delta.enableChangeDataFeed'")
    cdf_enabled = props.count() > 0 and props.first()["value"] == "true"
    status = "CDF enabled" if cdf_enabled else "CDF NOT enabled — sync may use snapshot mode"
    print(f"  {table}: {status}")

## Create Synced Tables

In [ ]:
# Synced table config: 3-level name (catalog.schema.table) + autoscale spec
# Gold tables are materialized views — must use SNAPSHOT (full-copy) mode
synced_table_configs = [
    {
        "source": f"{CATALOG}.dbx_res_gold.gold_portfolio_property_metrics",
        "synced_table_id": f"{CATALOG}.dbx_res_gold.portfolio_metrics",
        "pk": ["property_id"],
    },
    {
        "source": f"{CATALOG}.dbx_res_gold.gold_portfolio_time_series",
        "synced_table_id": f"{CATALOG}.dbx_res_gold.portfolio_time_series",
        "pk": ["rent_month"],
    },
]

for config in synced_table_configs:
    synced_table_id = config["synced_table_id"]
    print(f"\nCreating synced table: {synced_table_id}")
    print(f"  Source: {config['source']}")
    print(f"  Primary key: {config['pk']}")

    # Check if it already exists
    try:
        existing = w.api_client.do(
            "GET",
            f"/api/2.0/postgres/synced_tables/{synced_table_id}",
        )
        print(f"  Already exists: {existing.get('status', {}).get('detailed_state', 'unknown')}")
        continue
    except Exception:
        pass

    # Create via the postgres synced tables API (autoscale)
    payload = {
        "spec": {
            "source_table_full_name": config["source"],
            "project": f"projects/{PROJECT_ID}",
            "branch": f"projects/{PROJECT_ID}/branches/{BRANCH}",
            "primary_key_columns": config["pk"],
            "scheduling_policy": "SNAPSHOT",
            "postgres_database": DATABASE,
            "create_database_objects_if_missing": True,
        }
    }
    result = w.api_client.do(
        "POST",
        f"/api/2.0/postgres/synced_tables",
        query={"synced_table_id": synced_table_id},
        body=payload,
    )
    print(f"  Created: {result}")

## Check Sync Status

In [ ]:
import time

print("Waiting for sync to complete...")
for config in synced_table_configs:
    synced_table_id = config["synced_table_id"]
    for attempt in range(30):
        try:
            status = w.api_client.do(
                "GET",
                f"/api/2.0/postgres/synced_tables/{synced_table_id}",
            )
            state = status.get("status", {}).get("detailed_state", "UNKNOWN")
            if "ONLINE" in str(state) or "ACTIVE" in str(state):
                print(f"  {synced_table_id}: {state}")
                break
            elif "FAIL" in str(state) or "ERROR" in str(state):
                msg = status.get("status", {}).get("message", "")
                print(f"  {synced_table_id}: FAILED — {msg}")
                break
            else:
                if attempt % 5 == 0:
                    print(f"  {synced_table_id}: {state} (waiting...)")
                time.sleep(10)
        except Exception as e:
            print(f"  {synced_table_id}: Error checking status — {e}")
            time.sleep(10)
    else:
        print(f"  {synced_table_id}: timed out — check status manually")

print("\nSync setup complete.")